In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [2]:
# Import packages and modules
import numpy as np
import torch
from RL4CRN.iocrns.mass_action_iocrn import MassActionIOCRN
from RL4CRN.policies.add_reaction_from_library import AddReactionFromLibrary
from RL4CRN.agents.reinforce_agent import REINFORCEAgent
from RL4CRN.environments.environment import Environment
from RL4CRN.environments.parallel_environments import ParallelEnvironments
from RL4CRN.environments.serial_environments import SerialEnvironments

In [3]:
# Construct the basic CRN
species_labels = ['X_1', 'X_2', 'X_3']
inputs_labels = ['u_1', 'u_2', 'u_3', 'u_4']
S_R = np.array([[0], [1], [0]], dtype=np.int8)
S_P = np.array([[1], [1], [0]], dtype=np.int8)
k = 1
c = np.array([k], dtype=np.float32)
S_I = np.array([[0], [0], [0], [0]], dtype=np.int8)
o = np.array([1], dtype=np.int8)
iocrn_template = MassActionIOCRN(S_R, S_P, c, S_I, o, species_labels, inputs_labels)
iocrn_template.add_reaction({'reactant1 index': 0, 'reactant2 index': 1, 'product1 index': 0, 'product2 index': 0, 'input influence index': 0, 'rate constant':0.1}, mode='species index')
iocrn_template.add_reaction({'reactant1 index': 1, 'reactant2 index': 1, 'product1 index': 2, 'product2 index': 3, 'input influence index': 3, 'rate constant':0.5}, mode='species index')
iocrn_template.add_reaction({'reactant1 index': 1, 'reactant2 index': 3, 'product1 index': 0, 'product2 index': 3, 'input influence index': 2, 'rate constant':0.7}, mode='species index')
print('IOCRN template:')
print(iocrn_template)

IOCRN template:
Inputs: ['u_1', 'u_2', 'u_3', 'u_4'] 
Species: ['X_1', 'X_2', 'X_3'] 
Output Species: ['X_1'] 
Reaction 0: X_2 -> X_1 + X_2 ; Rate Constant: 1.0 
Reaction 1: X_1 -> 0 ; Rate Constant: 0.1 
Reaction 2: 2 X_1 -> X_2 + X_3 ; Rate Constant: 0.5u_3 
Reaction 3: X_1 + X_3 -> X_3 ; Rate Constant: 0.7u_2 



In [4]:
# Construct parallel environments
iocrn_0 = iocrn_template.clone()
max_num_reactions = 3
N = 5
N_CPUs = os.cpu_count()     
logger = None                                    
vec_env = ParallelEnvironments([Environment(iocrn_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(N)], N_CPUs=N_CPUs, logger=logger)
# vec_env = SerialEnvironments([Environment(iocrn_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(N)], logger=logger)

In [5]:
# Construct the Agent
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n = 3; p = 4
encoder_attributes = {"hidden_size": 64, "num_layers": 2}
structure_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
rate_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
input_influence_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
deep_layer_size = 1024
M = iocrn_0.get_reactions_range()
entropy_scheduler = {'initial_entropy_weight': 1, 'entropy_update_coefficient': 0.9, 'entropy_schedule': 20, 'minimum_entropy_weight': 1}
risk_scheduler = {'risk': 0.8, 'risk_update': 0.00, 'max_risk': 1.00, 'risk_schedule': 20}
policy = AddReactionFromLibrary(M, p, encoder_attributes, deep_layer_size, structure_decoder_attributes, rate_decoder_attributes, input_influence_decoder_attributes, continuous_distribution='lognormal', allow_input_influence=True)
Agent = REINFORCEAgent(policy, allow_input_influence=False, logger=logger, learning_rate=1e-5, entropy_scheduler=entropy_scheduler, risk_scheduler=risk_scheduler, device=device)

In [6]:
# Collect observations from the batch of CRNs
observation_batch = vec_env.observe()
reactions_indices_batch, rate_constants_batch, reactions_indices_influenced_by_inputs_batch = observation_batch
for i in range(N):
    print(f"Reactions Indices for IOCRN {i}: {reactions_indices_batch[i]}")
    print(f"Parameters for IOCRN {i}: {rate_constants_batch[i]}")
    for j in range(p):
        print(f"Reactions Indices influenced by input {j+1} for IOCRN {i}: {reactions_indices_influenced_by_inputs_batch[j][i]}")
    print("---------------------")

Reactions Indices for IOCRN 0: [23 10 44 58]
Parameters for IOCRN 0: [1.  0.1 0.5 0.7]
Reactions Indices influenced by input 1 for IOCRN 0: []
Reactions Indices influenced by input 2 for IOCRN 0: [58]
Reactions Indices influenced by input 3 for IOCRN 0: [44]
Reactions Indices influenced by input 4 for IOCRN 0: []
---------------------
Reactions Indices for IOCRN 1: [23 10 44 58]
Parameters for IOCRN 1: [1.  0.1 0.5 0.7]
Reactions Indices influenced by input 1 for IOCRN 1: []
Reactions Indices influenced by input 2 for IOCRN 1: [58]
Reactions Indices influenced by input 3 for IOCRN 1: [44]
Reactions Indices influenced by input 4 for IOCRN 1: []
---------------------
Reactions Indices for IOCRN 2: [23 10 44 58]
Parameters for IOCRN 2: [1.  0.1 0.5 0.7]
Reactions Indices influenced by input 1 for IOCRN 2: []
Reactions Indices influenced by input 2 for IOCRN 2: [58]
Reactions Indices influenced by input 3 for IOCRN 2: [44]
Reactions Indices influenced by input 4 for IOCRN 2: []
-----------

In [7]:
# Generate actions using the agent
actions = Agent.act(observation_batch)

# Visualize the actions
for i in range(N):
    print(f"Actions for IOCRN {i}:")
    print(f"Reaction Indices: {actions[i]['reaction index']} \t Reaction: X_{iocrn_0.map_reaction_to_species(actions[i]['reaction index'])[0]} + X_{iocrn_0.map_reaction_to_species(actions[i]['reaction index'])[1]} -> X_{iocrn_0.map_reaction_to_species(actions[i]['reaction index'])[2]} + X_{iocrn_0.map_reaction_to_species(actions[i]['reaction index'])[3]}")
    print(f"Reaction Rates: {actions[i]['rate constant']}")
    print(f"Input Influence Indices: {actions[i]['input influence index']}")
    print("---------------------")

Actions for IOCRN 0:
Reaction Indices: 49 	 Reaction: X_1 + X_2 -> X_0 + X_3
Reaction Rates: 0.4356766641139984
Input Influence Indices: 2
---------------------
Actions for IOCRN 1:
Reaction Indices: 70 	 Reaction: X_2 + X_2 -> X_1 + X_3
Reaction Rates: 0.16409263014793396
Input Influence Indices: 0
---------------------
Actions for IOCRN 2:
Reaction Indices: 52 	 Reaction: X_1 + X_2 -> X_2 + X_2
Reaction Rates: 0.5976499319076538
Input Influence Indices: 4
---------------------
Actions for IOCRN 3:
Reaction Indices: 76 	 Reaction: X_2 + X_3 -> X_0 + X_3
Reaction Rates: 0.7045122385025024
Input Influence Indices: 4
---------------------
Actions for IOCRN 4:
Reaction Indices: 4 	 Reaction: X_0 + X_0 -> X_1 + X_1
Reaction Rates: 0.22233854234218597
Input Influence Indices: 2
---------------------


In [8]:
# Take a step in the environment
vec_env.reset()
out = vec_env.step(actions, mode='reaction index')

# Print the results
for i in range(N):
    print(f"Results for IOCRN {i}:")
    print(vec_env.envs[i].state)
    print("---------------------")

Results for IOCRN 0:
Inputs: ['u_1', 'u_2', 'u_3', 'u_4'] 
Species: ['X_1', 'X_2', 'X_3'] 
Output Species: ['X_1'] 
Reaction 0: X_2 -> X_1 + X_2 ; Rate Constant: 1.0 
Reaction 1: X_1 -> 0 ; Rate Constant: 0.1 
Reaction 2: 2 X_1 -> X_2 + X_3 ; Rate Constant: 0.5u_3 
Reaction 3: X_1 + X_3 -> X_3 ; Rate Constant: 0.7u_2 
Reaction 4: X_1 + X_2 -> X_3 ; Rate Constant: 0.4356766641139984u_2 

---------------------
Results for IOCRN 1:
Inputs: ['u_1', 'u_2', 'u_3', 'u_4'] 
Species: ['X_1', 'X_2', 'X_3'] 
Output Species: ['X_1'] 
Reaction 0: X_2 -> X_1 + X_2 ; Rate Constant: 1.0 
Reaction 1: X_1 -> 0 ; Rate Constant: 0.1 
Reaction 2: 2 X_1 -> X_2 + X_3 ; Rate Constant: 0.5u_3 
Reaction 3: X_1 + X_3 -> X_3 ; Rate Constant: 0.7u_2 
Reaction 4: 2 X_2 -> X_1 + X_3 ; Rate Constant: 0.16409263014793396 

---------------------
Results for IOCRN 2:
Inputs: ['u_1', 'u_2', 'u_3', 'u_4'] 
Species: ['X_1', 'X_2', 'X_3'] 
Output Species: ['X_1'] 
Reaction 0: X_2 -> X_1 + X_2 ; Rate Constant: 1.0 
Reaction 